In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
from torchvision.datasets import Imagenette
import numpy as np
import tqdm
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to("cuda")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

In [ ]:
train_dataset = Imagenette(root = './data', split = 'train', download = True)
val_dataset = Imagenette(root = './data', split = 'val', download = True)

In [ ]:
def collate_fn(batch):
  images = [item[0] for item in batch]
  labels = [item[1] for item in batch]

  return images, labels

batch_size = 128

train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = False, collate_fn = collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False, collate_fn = collate_fn)

In [ ]:
train_features = []
train_labels = []

for (image, labels) in tqdm.tqdm(train_dataloader, desc = 'Total'):
  with torch.no_grad():
    inputs = processor(images = image, return_tensors = 'pt', padding = True).to("cuda")
    image_features = model.get_image_features(**inputs).pooler_output
    image_features = image_features / image_features.norm(dim = 1, keepdim = True)

    train_features.extend(image_features.cpu().tolist())
    train_labels.extend(labels)

In [ ]:
val_features = []
val_labels = []

for (image, labels) in tqdm.tqdm(val_dataloader, desc = 'Total'):
  with torch.no_grad():
    inputs = processor(images = image, return_tensors = 'pt', padding = True).to("cuda")
    image_features = model.get_image_features(**inputs).pooler_output
    image_features = image_features / image_features.norm(dim = 1, keepdim = True)

  val_features.extend(image_features.cpu().tolist())
  val_labels.extend(labels)

In [ ]:
LinearModel = nn.Linear(512, 10).to("cuda")
LossFunc = nn.CrossEntropyLoss()
optimizer = optim.Adam(LinearModel.parameters(), lr = 0.001)

In [ ]:
train_features = np.array(train_features)
train_labels = np.array(train_labels)
val_features = np.array(val_features)
val_labels = np.array(val_labels)

In [ ]:
train_features_tensor = torch.FloatTensor(train_features).to("cuda")
train_labels_tensor = torch.LongTensor(train_labels).to("cuda")
val_features_tensor = torch.FloatTensor(val_features).to("cuda")
val_labels_tensor = torch.LongTensor(val_labels).to("cuda")

In [ ]:
batch_size = 32
epochs = 15

print(train_features_tensor.shape)
print(train_labels_tensor.shape)

torch.Size([9469, 512])

torch.Size([9469])

In [ ]:
for epoch in range(epochs):
  idx = torch.randperm(len(train_features_tensor))
  for i in range(0, len(idx), batch_size):
    batch_idx = idx[i:i + batch_size]
    batch_features = train_features_tensor[batch_idx]
    batch_labels = train_labels_tensor[batch_idx]

    optimizer.zero_grad()
    outputs = LinearModel(batch_features)
    loss = LossFunc(outputs, batch_labels)
    loss.backward()
    optimizer.step()

with torch.no_grad():
  val_outputs = LinearModel(val_features_tensor)
  val_preds = torch.argmax(val_outputs, dim = 1)

  accuracy = accuracy_score(val_labels_tensor.cpu(), val_preds.cpu())

print(f"Accuracy = {100 * accuracy:.2f}%")

Accuracy = 99.59%

